# Using custom middleware to trim a long conversation

In [1]:
from dotenv import load_dotenv

# Remember: the .env file in the project folder must have the line
# OPENAI_API_KEY=sk-... (no quotes, no spaces).
load_dotenv()

True

#### Update note: current OpenAI models

* The original example is very good for learning the concept, and it worked with the OpenAI models available when the course was recorded: the agent used `gpt-4o-mini`.
* OpenAI has deprecated `gpt-4o-mini` (shutdown on January 14, 2027) and recommends `gpt-6-luna` as its replacement. We change it now so that the exercise keeps working after that date.
* This agent has no tools. The `ToolMessage` entries in the conversation are written by hand to simulate tool outputs, and the `trim_messages` middleware removes them before the model sees the conversation. So we only need to change the model name: `create_agent` still accepts it as text.
* `gpt-6-luna` reasons before it answers, so you may notice a short pause.
* The last question asks for the temperature of the device. That data (`temp=42C`) was only in a `ToolMessage`, which the middleware has removed. So the model cannot know it, and its answer will not include it. That is exactly what this example shows. With `gpt-6-luna`, the answer will be worded differently from the one in the video.
* The explanation below keeps the original code (`model="gpt-4o-mini"`).
* Check available models at developers.openai.com/api/docs/deprecations, and the current middleware options at docs.langchain.com.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, AIMessage
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

# Original version: this example used gpt-4o-mini.
# OpenAI has deprecated it (shutdown on January 14, 2027) and recommends
# gpt-6-luna as its replacement: small, fast and affordable for practice.
# This agent has no tools (the ToolMessages below are written by hand and the
# middleware removes them), so we only need to change the model name.
MODEL = "gpt-6-luna"

agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

print(response["messages"][-1].content)

#### If we print the detailed response with pprint, we will see that the tool messages have been trimmed from the conversation (short-term memory)

In [ ]:
from pprint import pprint

pprint(response)

## Let's explain the previous code in simple terms
Below is what this code is doing **in plain English**, **line by line**.

#### The main idea in one sentence

You’re creating an agent, and you’re adding a **middleware “filter”** that **removes ToolMessage entries** from the conversation **before the agent runs**, so the agent sees a “cleaner / shorter” chat history. (Middleware hooks like `before_agent` are meant for exactly this kind of state editing.)

---

#### Imports

```py
from langchain.agents import create_agent
```

* Imports `create_agent`, a helper that builds a ready-to-run **LLM agent**.

```py
from langgraph.checkpoint.memory import InMemorySaver
```

* Imports a **checkpointer** that can save the agent’s state (like messages) **in RAM** (not on disk). Useful for multi-turn conversations.

```py
from langchain.messages import HumanMessage, AIMessage
```

* Message objects:

  * `HumanMessage`: something the user said
  * `AIMessage`: something the model/agent said

```py
from typing import Any
```

* Python typing helper (`Any` means “could be any type”).

```py
from langchain.agents import AgentState
```

* `AgentState` is the “state dictionary type” the agent uses internally (it includes `"messages"`, plus other internal fields).

```py
from langchain.messages import RemoveMessage
```

* A special “instruction message” that tells LangGraph/LangChain: **remove a message with this ID** from the state.

```py
from langgraph.runtime import Runtime
```

* `Runtime` is extra runtime context passed into middleware hooks (think: metadata about the current run).

```py
from langchain.agents.middleware import before_agent
```

* Imports the middleware decorator that runs **once, right before the agent starts**.

```py
from langchain.messages import ToolMessage
```

* A `ToolMessage` represents output from a tool call (diagnostic logs, database results, etc.).

---

#### The middleware function (the “trimmer”)

```py
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
```

* `@before_agent` registers this function as middleware that runs **before the agent executes**. 
* The function receives:

  * `state`: the agent’s current state (includes the conversation messages)
  * `runtime`: info about this run (not used here, but available)
* It returns either:

  * a dict of updates to apply to the state (you return updates), or
  * `None` (meaning “don’t change anything”).

```py
    """Remove all the tool messages from the state"""
```

* A docstring explaining intent: delete tool outputs from the state.

```py
    messages = state["messages"]
```

* Pulls the message history list out of the agent state.

```py
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
```

* Scans through the message list and collects **only** the messages that are tool outputs.

```py
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}
```

* Returns a **state update** telling the system:

  * “For the `messages` field, apply these removals.”
* For each tool message found, you create a `RemoveMessage(id=...)` instruction to remove it.

**Effect:** when the agent begins, it will start with a message list where every `ToolMessage` has been deleted.

####  Important subtlety

Because we used **`before_agent`**, this runs **once per invocation** (one “agent run”).
If you needed to trim **before each model call inside the agent loop**, LangChain also provides `before_model` middleware (commonly used for message trimming).

---

#### Creating the agent

```py
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)
```

* `model="gpt-4o-mini"`: chooses the chat model the agent will use.
* `checkpointer=InMemorySaver()`:

  * enables saving/loading the agent’s state between runs (in memory).
* `middleware=[trim_messages]`:

  * attaches your middleware so it runs at the hook point. LangChain agents support a list of middleware like this.

---

#### Invoking the agent with a long conversation

```py
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)
```

#### First argument: the input state

* You pass an initial state with `"messages": [...]`.
* The list includes:

  * Human messages (user)
  * AI messages (assistant)
  * Tool messages (tool output)

#### What your middleware does right here

Before the agent starts, `trim_messages` runs and **removes the two `ToolMessage(...)` entries**. So the agent will effectively see something like:

1. User: “My device won’t turn on…”
2. AI: “Is it plugged in…”
3. User: “Yes…”
4. AI: “Any lights…”
5. User: “What’s the temperature…?”

That makes the context shorter and removes noisy tool logs.

#### Second argument: config with `thread_id`

```py
{"configurable": {"thread_id": "2"}}
```

* Because you’re using a checkpointer, LangGraph uses a **thread** to store the “accumulated state” across runs.
* `thread_id="2"` says: “Save/load state under thread 2.”

---

#### Printing the last assistant message

```py
print(response["messages"][-1].content)
```

* `response` is a state-like dict that includes `"messages"`.
* `[-1]` means “last message”.
* `.content` prints the text of that last message (the agent’s latest reply).

---

#### Quick mental model (beginner-friendly)

* **AgentState** = a backpack containing `"messages"` and other internal fields.
* **Middleware** = a checkpoint where you can open the backpack and remove/add things.
* `before_agent` = “do this cleanup once before the run starts.”
* `RemoveMessage` = a delete instruction that removes specific message IDs.
* `InMemorySaver + thread_id` = “remember the backpack for next time under this conversation ID.”

## How to run this code from Visual Studio Code
* Open Terminal.
* Make sure you are in the project folder.
* Make sure you have the poetry env activated.
* Enter and run the following command:
    * `python 012-mid-to-trim-convesation.py`